# Day 11 - ML Pipelines

This notebook contains all tasks from Day 11 covering sklearn pipelines, column transformers, and model deployment.

## Task 9: Titanic Pipeline with GridSearch

In [ ]:
import pandas as pd
import seaborn as sns
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier

df = sns.load_dataset('titanic')

df = df.drop(columns=['deck', 'embark_town', 'alive', 'who', 'adult_male', 'class'])

X = df.drop('survived', axis=1)
y = df['survived']

num_cols = ['age', 'fare', 'parch', 'sibsp']
cat_cols = ['sex', 'embarked']
ord_cols = ['pclass']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder())
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
    ('ord', ord_pipeline, ord_cols)
])

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')

print(f"CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [5, 10, None]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X, y)

print("Best Params:", grid.best_params_)
print("Best CV Score:", round(grid.best_score_, 4))

joblib.dump(grid.best_estimator_, 'titanic_pipeline.pkl')

loaded_model = joblib.load('titanic_pipeline.pkl')

new_passenger = pd.DataFrame([{
    'pclass': 3,
    'sex': 'male',
    'age': 30,
    'sibsp': 0,
    'parch': 0,
    'fare': 7.25,
    'embarked': 'S'
}])

prediction = loaded_model.predict(new_passenger)[0]
probability = loaded_model.predict_proba(new_passenger)[0][1]

print("Survived Prediction:", prediction)
print("Survival Probability:", round(probability, 4))